# S11 LAST 70% + S8C Full-MRI Direct 30% — LB Submission V1

## 목적

사용자 확인 Public LB:

```text
S10 standalone                         0.923
S10 LAST 70 + S8C Direct 30           0.926
S11 LAST 2F standalone                0.926
```

이번 제출은 S11 standalone `0.926`에, S10에서 이미 +0.003 gain을 보였던
**S8C Full-MRI Direct 30%**를 동일한 방식으로 결합한다.

정상 경로:

```text
hidden-test raw MRI
        ↓
Stage1-B Strict-FT DINO Base F0/F1
        ↓
Full-MRI features
        ↓
S8C F0/F1
  ├─ Full-MRI Direct probability 2F mean ───────────────┐
  └─ label-free Top18 selector                          │ 30%
                    ↓                                   │
        selected raw DICOM windows                      │
                    ↓                                   │
 DETAIL 110mm/280 + ANCHOR 130mm/224 + CONTEXT 160mm/224
                    ↓
           S11 LAST F0/F1
                    ↓
          probability 2F mean
                    │ 70%
                    └──────────────┬─────────────────────┘
                                   ↓
                          0.70 × S11
                        + 0.30 × S8C Direct
                                   ↓
                           submission.csv
```

## 중요한 점

- **S11 standalone 코드의 hidden-test DICOM / selector / Multi-View 경로를 그대로 사용한다.**
- S8C Direct는 selector 계산 과정에서 이미 같은 forward로 산출되는 probability다.
- 새로운 모델/feature extraction을 추가하지 않는다.
- probability space blend다.
- S11 fold mean = 살아있는 S11 Fold들의 probability mean.
- S8C Direct fold mean = 살아있는 S8C Fold들의 probability mean.
- 정상 경로는 두 branch 모두 2F일 때:
  `0.70*S11_LAST_2F + 0.30*S8C_DIRECT_2F`.
- 정상 Study의 blend 산술 오차를 `<1e-12`로 검증한다.
- Study-level `0.5` placeholder는 사용하지 않는다.

## Rescue

정상 hidden-test에서는 rescue가 없어야 한다.

- S11 한 Fold만 실패 → 살아있는 S11 Fold 사용
- S8C 한 Fold만 실패 → 살아있는 S8C Fold의 Direct와 selector 사용
- S8C 두 Fold 실패 → deterministic label-free Top18 selector로 S11은 계속 시도하지만 Direct branch는 없음
- S11 branch만 살아있으면 S11 100%
- S8C Direct branch만 살아있으면 S8C Direct 100%
- 둘 다 실제 이미지 예측이 없으면 notebook을 실패시킴
- **Study-level neutral 0.5 fallback 없음**

## 필수 Kaggle Input

S11 standalone과 동일하다.

1. Competition dataset
2. MetaResearch DINOv2 Base
3. `woniyeon/rsna-knee-s1ft02-b-v4strict-2fold-best-v1`
4. `yhlucas/s8c-hier-attn-mil`
5. S11 checkpoint dataset:
   `S11_TOP18_MULTIVIEW_AUG_FT_V1/fold0/last.pt`
   `S11_TOP18_MULTIVIEW_AUG_FT_V1/fold1/last.pt`

## 실행

- GPU: **T4 ×2**
- Internet: **OFF**
- Save & Run All
- 추천 Save Version: `S11-LAST70-S8C30-LB-V1`
- output: `/kaggle/working/submission.csv`

## 비교 기준

이번 Public LB를 다음과 비교한다.

```text
S10 standalone                 0.923
S10 70 + S8C 30               0.926
S11 standalone                 0.926
S11 70 + S8C 30               이번 제출
```

In [ ]:
from __future__ import annotations
import gc, hashlib, json, math, os, re, time, warnings, zipfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
from torch import nn
from transformers import Dinov2Config, Dinov2Model

warnings.filterwarnings("ignore")

EXPERIMENT = "S11-LAST-70-S8C-DIRECT-30-2F-LB-SUBMISSION-V1"
BLEND_S11 = 0.70
BLEND_S8C = 0.30
assert abs(BLEND_S11 + BLEND_S8C - 1.0) < 1e-12
UID = "StudyInstanceUID"
SID = "SeriesInstanceUID"
TARGETS = [
    'ACL','MCL','Medial Meniscus','Lateral Meniscus',
    'Medial OA','Lateral OA','PF OA','Effusion',
    'Synovitis',"Baker's",'Contusion','Fracture'
]
TOPK = 18
MIN_GAP = 3

DETAIL_FOV_MM = 110.0
DETAIL_SIZE = 280
ANCHOR_FOV_MM = 130.0
ANCHOR_SIZE = 224
CONTEXT_FOV_MM = 160.0
CONTEXT_SIZE = 224

ROUTING_NP = np.asarray([
    [.65,.30,.05],
    [.05,.30,.65],
    [.60,.35,.05],
    [.55,.40,.05],
    [.65,.30,.05],
    [.60,.35,.05],
    [.50,.40,.10],
    [.05,.30,.65],
    [.10,.40,.50],
    [.00,.15,.85],
    [.15,.45,.40],
    [.40,.45,.15],
], dtype=np.float32)
assert ROUTING_NP.shape == (12,3)
assert np.allclose(ROUTING_NP.sum(1), 1.0)

STAGE1_ROOT = Path(
    '/kaggle/input/datasets/woniyeon/'
    'rsna-knee-s1ft02-b-v4strict-2fold-best-v1'
)
S8C_ROOT = Path(
    '/kaggle/input/datasets/yhlucas/s8c-hier-attn-mil'
)
S11_ROOT_CANDIDATES = [
    Path('/kaggle/input/rsna-knee-checkpoint-models/S11_TOP18_MULTIVIEW_AUG_FT_V1'),
    Path('/kaggle/input/datasets/woniyeon/rsna-knee-checkpoint-models/S11_TOP18_MULTIVIEW_AUG_FT_V1'),
    Path('/kaggle/input/datasets/yhlucas/rsna-knee-checkpoint-models/S11_TOP18_MULTIVIEW_AUG_FT_V1'),
]

STAGE1_SHA = {
    0:'b4e7d29d3ae0e7ff999c3fb0119dc6126ddd5d5a006c69868818bc440c895aa4',
    1:'966fdfe174c995bfbd8855b520440c508a6aa8539eee9cb5a5f724402b538d79',
}
S8C_SHA = {
    0:'1b16bd88d3a5a13d93bbfb45ac12561e516a17cbf8d34fab50204bf09a67cfa8',
    1:'506e73b7462195b3091b60508aa8781200f1d72c8c0469d8cea57a8b11771155',
}
S11_LAST_SHA = {
    0:'2c962f22a1ebc55de81ee674c0f8e9b2c69001f028bc3db19dd6b114aa8f2dec',
    1:'6647156f1bad9ed073b135ebea9c5bd4452d9483a17186c69a1dd04d22ae40bf',
}
S10_INIT_SHA = {
    0:'e30d268a039dbf64a6ba082d4c85cdb1e1ed2090c70e59d9165b3cfe83083908',
    1:'02232b0bdcf8062564d04feda43f9fe6ea09edd1553a7025caaf19ac701ee1b3',
}
S9_TOP18_TRAIN_SHA = (
    '51c0ee5aa02e87c1e15e85d38596dfcd7da8d22c81ceefa9046a67990df3e961'
)

OUT = Path('/kaggle/working')
AUDIT = OUT / 'S11_LAST_70_S8C_DIRECT_30_V1_QA'
AUDIT.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(8<<20),b''):
            h.update(b)
    return h.hexdigest()

def find_competition_root():
    for root in [
        Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
        Path('/kaggle/input/rsna-knee-abnormality-detection'),
    ]:
        if (
            (root/'test.csv').is_file()
            and (root/'test_series.csv').is_file()
            and (root/'sample_submission.csv').is_file()
            and (root/'test_series').is_dir()
        ):
            return root
    raise FileNotFoundError('Competition input not found')

def find_dino_root():
    for root in [
        Path('/kaggle/input/models/metaresearch/dinov2/pytorch/base/1'),
        Path('/kaggle/input/models/metaresearch/dinov2/PyTorch/base/1'),
        Path('/kaggle/input/metaresearch/dinov2/pytorch/base/1'),
    ]:
        if (root/'config.json').is_file():
            return root
    raise FileNotFoundError('MetaResearch DINOv2 Base config not found')

def one_dir(cands,name):
    hits=[p for p in cands if p.is_dir()]
    assert len(hits)==1,(name,hits,cands)
    return hits[0]

assert torch.cuda.is_available() and torch.cuda.device_count()>=2, 'T4 x2 required'
assert all(
    'T4' in torch.cuda.get_device_name(i)
    for i in (0,1)
), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]

COMP=find_competition_root()
DINO_ROOT=find_dino_root()
DICOM_ROOT=COMP/'test_series'
S11_ROOT=one_dir(S11_ROOT_CANDIDATES,'S11 checkpoint root')

for f in (0,1):
    p=STAGE1_ROOT/f'fold{f}'/'last.pt'
    assert p.is_file() and sha256(p)==STAGE1_SHA[f], ('STAGE1_SHA',f,p)
    p=S8C_ROOT/f'fold{f}'/'best.pt'
    assert p.is_file() and sha256(p)==S8C_SHA[f], ('S8C_SHA',f,p)
    p=S11_ROOT/f'fold{f}'/'last.pt'
    assert p.is_file() and sha256(p)==S11_LAST_SHA[f], (
        'S11_LAST_SHA',f,p,sha256(p) if p.is_file() else None
    )

TEST=pd.read_csv(COMP/'test.csv',dtype={UID:str})
TS=pd.read_csv(COMP/'test_series.csv',dtype={UID:str,SID:str})
SAMPLE=pd.read_csv(COMP/'sample_submission.csv',dtype={UID:str})
assert SAMPLE[UID].is_unique and set(SAMPLE[UID])==set(TEST[UID])
assert list(SAMPLE.columns)==[UID]+TARGETS
assert not TS[[UID,SID]].duplicated().any()
assert not (set(SAMPLE[UID])-set(TS[UID]))

cfg=Dinov2Config.from_json_file(str(DINO_ROOT/'config.json'))
assert cfg.hidden_size==768 and cfg.num_hidden_layers==12
assert cfg.num_attention_heads==12 and cfg.patch_size==14

print(
    'PREFLIGHT_PASS',EXPERIMENT,
    'studies',len(SAMPLE),
    'series',len(TS),
    'S11_ROOT',S11_ROOT,
    'GPUs',[torch.cuda.get_device_name(i) for i in (0,1)]
)

In [ ]:
# Exact selector + S11 model definitions.
def build_base_encoder():
    cfg=Dinov2Config.from_json_file(str(DINO_ROOT/'config.json'))
    return Dinov2Model(cfg)

class S8CModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj=nn.Sequential(
            nn.LayerNorm(1536),nn.Linear(1536,256),
            nn.GELU(),nn.Dropout(.1)
        )
        self.window_attn=nn.Linear(256,12)
        self.head=nn.Linear(256,12)
        self.series_query=nn.Parameter(torch.zeros(12,256))
        self.series_scale=1/math.sqrt(256.)

class S11FeatureDINO(nn.Module):
    def __init__(self, routing):
        super().__init__()
        cfg=Dinov2Config.from_json_file(str(DINO_ROOT/'config.json'))
        self.encoder=Dinov2Model(cfg)
        self.feature_proj=nn.Sequential(
            nn.LayerNorm(1536),nn.Linear(1536,256),
            nn.GELU(),nn.Dropout(.10)
        )
        self.target_attn=nn.Linear(256,12)
        self.target_weight=nn.Parameter(torch.empty(12,256))
        self.target_bias=nn.Parameter(torch.zeros(12))
        self.mean_residual=nn.Linear(256,12)
        self.register_buffer(
            'routing',
            torch.as_tensor(routing,dtype=torch.float32),
            persistent=False,
        )

    def _features(self,x):
        o=self.encoder(
            pixel_values=x,
            interpolate_pos_encoding=True,
            return_dict=True,
        ).last_hidden_state
        return torch.cat([o[:,0],o[:,1:].mean(1)],dim=-1)

    def one_view(self,images,chunk=18):
        assert images.ndim==5 and images.shape[1]==18
        B,W,C,H,I=images.shape
        flat=images.reshape(B*W,C,H,I)
        parts=[]
        for i in range(0,len(flat),chunk):
            parts.append(self._features(flat[i:i+chunk]))
        z=self.feature_proj(torch.cat(parts,0)).reshape(B,W,256)
        attn=torch.softmax(
            self.target_attn(z).transpose(1,2).float(),
            dim=-1,
        ).to(z.dtype)
        pooled=torch.bmm(attn,z)
        logits=(
            pooled*self.target_weight[None]
        ).sum(-1)+self.target_bias[None]
        return logits + .1*self.mean_residual(z.mean(1))

    def forward(self,detail,anchor,context,chunk=18):
        ld=self.one_view(detail,chunk)
        la=self.one_view(anchor,chunk)
        lc=self.one_view(context,chunk)
        r=self.routing.to(dtype=ld.dtype,device=ld.device)
        return ld*r[:,0] + la*r[:,1] + lc*r[:,2]

SELECTOR_ENCODERS=[]
S8C_MODELS=[]
S11_MODELS=[]
MODEL_AUDIT=[]

for f in (0,1):
    dev=torch.device(f'cuda:{f}')

    enc=build_base_encoder()
    st=torch.load(
        STAGE1_ROOT/f'fold{f}'/'last.pt',
        map_location='cpu',
        weights_only=False,
    )
    assert int(st['fold'])==f
    enc_sd={
        str(k)[8:]:v
        for k,v in st['model'].items()
        if str(k).startswith('encoder.')
    }
    res=enc.load_state_dict(enc_sd,strict=True)
    assert not res.missing_keys and not res.unexpected_keys
    enc=enc.to(dev).eval()

    s8=S8CModel().to(dev).eval()
    ck=torch.load(
        S8C_ROOT/f'fold{f}'/'best.pt',
        map_location='cpu',
        weights_only=False,
    )
    assert ck['fold']==f and ck['mode']=='HIER_ATTN'
    assert ck['experiment']=='S8C'
    assert ck['targets']==TARGETS
    assert ck['feature_checkpoint_sha256']==STAGE1_SHA[f]
    res=s8.load_state_dict(ck['model'],strict=True)
    assert not res.missing_keys and not res.unexpected_keys

    s11=S11FeatureDINO(ROUTING_NP).to(dev).eval()
    fc=torch.load(
        S11_ROOT/f'fold{f}'/'last.pt',
        map_location='cpu',
        weights_only=False,
    )
    assert int(fc['fold'])==f
    assert fc.get('stage')=='S11_TOP18_MULTIVIEW_AUG_FT'
    assert fc.get('selector_top18_sha256')==S9_TOP18_TRAIN_SHA
    assert fc.get('init_checkpoint_sha256')==S10_INIT_SHA[f]
    assert np.allclose(
        np.asarray(fc.get('routing'),dtype=np.float32),
        ROUTING_NP,
        atol=1e-7,
    )
    res=s11.load_state_dict(fc['model'],strict=True)
    assert not res.missing_keys and not res.unexpected_keys

    SELECTOR_ENCODERS.append(enc)
    S8C_MODELS.append(s8)
    S11_MODELS.append(s11)
    MODEL_AUDIT.append({
        'fold':f,
        'stage1_last_sha':STAGE1_SHA[f],
        's8c_sha':S8C_SHA[f],
        's11_last_sha':S11_LAST_SHA[f],
        's11_epoch':int(fc['epoch']),
        's11_metric_auc':float(fc.get('metric_auc',float('nan'))),
    })

    del st,enc_sd,ck,fc
    gc.collect()
    print('MODEL_LOAD_PASS',f,'S11=LAST','S8C=SELECTOR')

In [ ]:
# Hidden-test robust DICOM loader. Normal path reproduces RMIL04B Native224 V3.
HEADER_TAGS=['StudyInstanceUID','SeriesInstanceUID','SOPInstanceUID','ImageOrientationPatient','ImagePositionPatient','InstanceNumber','PixelSpacing','Rows','Columns','PhotometricInterpretation','RescaleSlope','RescaleIntercept']

def natural_key(path):
    parts=re.split(r'(\d+)',Path(path).name); return tuple(int(x) if x.isdigit() else x.lower() for x in parts)

def safe_float(value):
    try:
        x=float(value); return x if np.isfinite(x) else None
    except Exception:return None

def safe_array(value,n):
    try:
        a=np.asarray(value,dtype=np.float64).reshape(-1)
        if len(a)>=n and np.isfinite(a[:n]).all(): return a[:n]
    except Exception:pass
    return None

def valid_normal(iop):
    if iop is None:return None
    try:
        n=np.cross(iop[:3],iop[3:6]); z=float(np.linalg.norm(n))
        return None if (not np.isfinite(z) or z<1e-8) else n/z
    except Exception:return None

def read_header_record(path,idx,uid,sid):
    r={'path':Path(path),'original_index':int(idx),'header_ok':False,'instance_number':None,'ipp':None,'normal':None,'rows':None,'cols':None,'spacing':None,'sop':Path(path).name}
    try:
        ds=pydicom.dcmread(path,stop_before_pixels=True,force=True,specific_tags=HEADER_TAGS); r['header_ok']=True
        r['sop']=str(getattr(ds,'SOPInstanceUID','') or Path(path).name); r['instance_number']=safe_float(getattr(ds,'InstanceNumber',None))
        r['ipp']=safe_array(getattr(ds,'ImagePositionPatient',None),3); iop=safe_array(getattr(ds,'ImageOrientationPatient',None),6); r['normal']=valid_normal(iop)
        rows=safe_float(getattr(ds,'Rows',None)); cols=safe_float(getattr(ds,'Columns',None));
        if rows is not None and cols is not None and rows>=1 and cols>=1:r['rows'],r['cols']=int(rows),int(cols)
        ps=safe_array(getattr(ds,'PixelSpacing',None),2)
        if ps is not None and (ps>0).all():r['spacing']=(float(ps[0]),float(ps[1]))
    except Exception:pass
    return r

def read_series_headers(uid,sid):
    root=DICOM_ROOT/uid/sid
    if not root.is_dir():raise FileNotFoundError(str(root))
    files=sorted(root.glob('*.dcm'),key=natural_key)
    if not files:raise RuntimeError(f'No DICOM files: {uid}/{sid}')
    rec=[read_header_record(p,i,uid,sid) for i,p in enumerate(files)]
    ref=next((r['normal'] for r in rec if r['normal'] is not None),None)
    for r in rec:r['spatial_position']=float(np.dot(r['ipp'],ref)) if ref is not None and r['ipp'] is not None else None
    n=len(rec); need=max(2,int(math.ceil(.80*n))); spatial=sum(r['spatial_position'] is not None for r in rec); inst=sum(r['instance_number'] is not None for r in rec)
    if spatial>=need:
        vals=[r['spatial_position'] for r in rec if r['spatial_position'] is not None]; fb=float(np.median(vals)); rec.sort(key=lambda r:(r['spatial_position'] if r['spatial_position'] is not None else fb,r['instance_number'] if r['instance_number'] is not None else float('inf'),r['sop'],r['original_index'])); method='IOP_IPP'
    elif inst>=need:
        rec.sort(key=lambda r:(r['instance_number'] if r['instance_number'] is not None else float('inf'),r['original_index'])); method='InstanceNumber'
    else:
        rec.sort(key=lambda r:natural_key(r['path'])); method='filename'
    shapes=[(r['rows'],r['cols']) for r in rec if r['rows'] and r['cols']]; spacings=[r['spacing'] for r in rec if r['spacing'] is not None]
    return rec,(Counter(shapes).most_common(1)[0][0] if shapes else None),(Counter(spacings).most_common(1)[0][0] if spacings else None),{'sort_method':method,'header_failures':sum(not r['header_ok'] for r in rec)}

def crop_geometry(shape,spacing):
    h,w=map(int,shape); fallback=False
    if spacing is None or len(spacing)!=2 or not np.isfinite(spacing).all() or min(spacing)<=0:nh,nw=h,w;fallback=True
    else:
        dy,dx=map(float,spacing);nh=int(round(130./dy));nw=int(round(130./dx))
        if nh<3 or nw<3:nh,nw=h,w;fallback=True
    y=int(round((h-nh)/2.));x=int(round((w-nw)/2.));y0,x0=max(0,y),max(0,x);y1,x1=min(h,y+nh),min(w,x+nw)
    if not(0<=y0<y1<=h and 0<=x0<x1<=w):y0,x0,y1,x1=0,0,h,w;nh,nw=h,w;y=x=0;fallback=True
    pads=(y0-y,y+nh-y1,x0-x,x+nw-x1)
    if min(pads)<0:pads=(0,0,0,0);fallback=True
    return dict(y0=y0,x0=x0,y1=y1,x1=x1,pad_top=int(pads[0]),pad_bottom=int(pads[1]),pad_left=int(pads[2]),pad_right=int(pads[3]),want_h=int(nh),want_w=int(nw),has_padding=bool(any(pads)),geometry_fallback=bool(fallback))

def decode_rescaled_frame(path):
    ds=pydicom.dcmread(path,force=True); arr=np.squeeze(np.asarray(ds.pixel_array))
    if arr.ndim!=2:raise ValueError(arr.shape)
    arr=arr.astype(np.float32,copy=False); slope=safe_float(getattr(ds,'RescaleSlope',1.)); intercept=safe_float(getattr(ds,'RescaleIntercept',0.)); slope=1. if slope is None else slope; intercept=0. if intercept is None else intercept
    frame=arr*slope+intercept
    if str(getattr(ds,'PhotometricInterpretation','MONOCHROME2') or 'MONOCHROME2').upper()=='MONOCHROME1':frame=-frame
    ps=safe_array(getattr(ds,'PixelSpacing',None),2); spacing=((float(ps[0]),float(ps[1])) if ps is not None and (ps>0).all() else None)
    return frame,spacing

def build_series_windows(uid,sid):
    records,shape,spacing,diag=read_series_headers(uid,sid); decoded=[]; decsp=[]; errors=0
    for r in records:
        try:
            frame,sp=decode_rescaled_frame(r['path']); shape=tuple(map(int,frame.shape)) if shape is None else shape
            if tuple(frame.shape)!=tuple(shape):frame=cv2.resize(frame,(int(shape[1]),int(shape[0])),interpolation=cv2.INTER_LINEAR);diag['pixel_shape_resized']=diag.get('pixel_shape_resized',0)+1
            decoded.append(frame.astype(np.float32,copy=False));
            if sp is not None:decsp.append(sp)
        except Exception:decoded.append(None);errors+=1
    valid=[i for i,x in enumerate(decoded) if x is not None]
    if not valid:raise RuntimeError(f'All slices failed: {uid}/{sid}')
    for i,x in enumerate(decoded):
        if x is None:decoded[i]=decoded[min(valid,key=lambda z:abs(z-i))].copy()
    if shape is None:shape=tuple(map(int,decoded[valid[0]].shape))
    if spacing is None and decsp:spacing=Counter(decsp).most_common(1)[0][0]
    geo=crop_geometry(shape,spacing); data=np.stack([x[geo['y0']:geo['y1'],geo['x0']:geo['x1']] for x in decoded],0).astype(np.float32)
    bad=~np.isfinite(data)
    if bad.any():
        finite=data[~bad];data[bad]=float(np.median(finite)) if finite.size else 0.
    flat=data.reshape(-1); stride=max(1,int(math.ceil(flat.size/200000))); lo,hi=np.percentile(flat[::stride],[.5,99.5]);deg=False
    if np.isfinite(lo) and np.isfinite(hi) and hi>lo:np.clip(data,lo,hi,out=data)
    else:deg=True
    mu=float(np.mean(data,dtype=np.float64));sd=float(np.std(data,dtype=np.float64))
    if not np.isfinite(sd) or sd<=1e-6:sd=1.;deg=True
    pads=((geo['pad_top'],geo['pad_bottom']),(geo['pad_left'],geo['pad_right'])); vol=np.empty((len(decoded),224,224),np.uint8)
    for i in range(len(decoded)):
        x=np.clip((data[i]-mu)/sd,-5.,5.).astype(np.float32)
        if geo['has_padding']:x=np.pad(x,pads,mode='constant',constant_values=-5.)
        if x.shape!=(geo['want_h'],geo['want_w']):x=cv2.resize(x,(geo['want_w'],geo['want_h']),interpolation=cv2.INTER_LINEAR)
        x=cv2.resize(x,(224,224),interpolation=cv2.INTER_AREA);vol[i]=np.rint(np.clip((x+5.)*25.5,0,255)).astype(np.uint8)
    n=len(vol)
    if n>=3:
        windows=np.stack([vol[:-2],vol[1:-1],vol[2:]],axis=1);centers=np.arange(1,n-1,dtype=np.int32)
    else:
        ws=[];cs=[]
        for j in range(n):ws.append(np.stack([vol[max(0,j-1)],vol[j],vol[min(n-1,j+1)]],0));cs.append(j)
        windows=np.stack(ws,0);centers=np.asarray(cs,np.int32)
    windows=np.ascontiguousarray(windows,dtype=np.uint8);assert windows.shape[1:]==(3,224,224)
    diag.update({UID:uid,SID:sid,'slices':n,'windows':len(windows),'decode_errors':errors,'geometry_fallback':int(geo['geometry_fallback']),'short_series_edge_replication':int(n<3),'degenerate_intensity_fallback':int(deg)})
    return windows,centers,diag


# ------------------------------------------------------------------
# S11 exact multi-view reconstruction for SELECTED Top18 positions.
# Normal path reproduces the training-cache preprocessing.
# ------------------------------------------------------------------
def decode_series_raw_robust(uid,sid):
    records,shape,spacing,diag=read_series_headers(uid,sid)
    decoded=[]
    decsp=[]
    errors=0

    for r in records:
        try:
            frame,sp=decode_rescaled_frame(r['path'])
            if shape is None:
                shape=tuple(map(int,frame.shape))
            if tuple(frame.shape)!=tuple(shape):
                frame=cv2.resize(
                    frame,
                    (int(shape[1]),int(shape[0])),
                    interpolation=cv2.INTER_LINEAR,
                )
                diag['pixel_shape_resized']=diag.get(
                    'pixel_shape_resized',0
                )+1
            decoded.append(frame.astype(np.float32,copy=False))
            if sp is not None:
                decsp.append(sp)
        except Exception:
            decoded.append(None)
            errors+=1

    valid=[i for i,x in enumerate(decoded) if x is not None]
    if not valid:
        raise RuntimeError(f'All slices failed: {uid}/{sid}')

    for i,x in enumerate(decoded):
        if x is None:
            decoded[i]=decoded[
                min(valid,key=lambda z:abs(z-i))
            ].copy()

    if shape is None:
        shape=tuple(map(int,decoded[valid[0]].shape))

    if spacing is None and decsp:
        spacing=Counter(decsp).most_common(1)[0][0]

    raw=np.stack(decoded,axis=0).astype(np.float32)
    bad=~np.isfinite(raw)
    if bad.any():
        finite=raw[~bad]
        raw[bad]=float(np.median(finite)) if finite.size else 0.0

    return raw,shape,spacing,{
        **diag,
        'multiview_decode_errors':errors,
    }

def no_pad_geometry_robust(shape,spacing,requested_fov_mm,out_size):
    h,w=map(int,shape)
    fallback=False

    if (
        spacing is None
        or len(spacing)!=2
        or not np.isfinite(spacing).all()
        or min(spacing)<=0
    ):
        real_h,real_w=h,w
        want_h,want_w=h,w
        fallback=True
        source_fov_y=source_fov_x=None
        cov_y=cov_x=1.0
    else:
        dy,dx=map(float,spacing)
        want_h=max(3,int(round(requested_fov_mm/dy)))
        want_w=max(3,int(round(requested_fov_mm/dx)))
        real_h=min(h,want_h)
        real_w=min(w,want_w)
        source_fov_y=float(h*dy)
        source_fov_x=float(w*dx)
        cov_y=min(1.0,float(real_h*dy)/requested_fov_mm)
        cov_x=min(1.0,float(real_w*dx)/requested_fov_mm)

    y0=int(round((h-real_h)/2.0))
    x0=int(round((w-real_w)/2.0))
    y1=y0+real_h
    x1=x0+real_w

    assert 0<=y0<y1<=h
    assert 0<=x0<x1<=w

    return {
        'requested_fov_mm':float(requested_fov_mm),
        'out_size':int(out_size),
        'want_h':int(want_h),'want_w':int(want_w),
        'real_h':int(real_h),'real_w':int(real_w),
        'y0':int(y0),'x0':int(x0),'y1':int(y1),'x1':int(x1),
        'coverage_y':float(cov_y),'coverage_x':float(cov_x),
        'coverage_min':float(min(cov_y,cov_x)),
        'source_fov_mm_y':source_fov_y,
        'source_fov_mm_x':source_fov_x,
        'geometry_fallback':int(fallback),
    }

def normalized_view_selected(raw,geom,indices):
    y0,y1=geom['y0'],geom['y1']
    x0,x1=geom['x0'],geom['x1']
    cropped=raw[:,y0:y1,x0:x1].astype(np.float32,copy=True)
    assert cropped.ndim==3 and np.isfinite(cropped).all()

    flat=cropped.reshape(-1)
    stride=max(1,int(math.ceil(flat.size/200000)))
    lo,hi=np.percentile(flat[::stride],[.5,99.5])

    degenerate=False
    if np.isfinite(lo) and np.isfinite(hi) and hi>lo:
        np.clip(cropped,lo,hi,out=cropped)
    else:
        degenerate=True

    mu=float(np.mean(cropped,dtype=np.float64))
    sd=float(np.std(cropped,dtype=np.float64))
    if not np.isfinite(sd) or sd<=1e-6:
        sd=1.0
        degenerate=True

    result={}
    for idx in sorted(set(int(x) for x in indices)):
        assert 0<=idx<len(cropped)
        x=np.clip((cropped[idx]-mu)/sd,-5.,5.).astype(np.float32)
        x=cv2.resize(
            x,
            (geom['out_size'],geom['out_size']),
            interpolation=cv2.INTER_AREA,
        )
        result[idx]=np.rint(
            np.clip((x+5.0)*25.5,0,255)
        ).astype(np.uint8)

    return result,{
        'norm_sample_stride':int(stride),
        'norm_p005':float(lo),
        'norm_p995':float(hi),
        'norm_mean':mu,
        'norm_std':sd,
        'degenerate_intensity_fallback':int(degenerate),
    }

def make_selected_triplets(slice_map,centers):
    return np.stack([
        np.stack([
            slice_map[int(c)-1],
            slice_map[int(c)],
            slice_map[int(c)+1],
        ],axis=0)
        for c in centers
    ],axis=0)

def anchor_resize_fallback(anchor_windows,out_size):
    result=np.empty(
        (len(anchor_windows),3,out_size,out_size),
        dtype=np.uint8,
    )
    for i in range(len(anchor_windows)):
        for c in range(3):
            result[i,c]=cv2.resize(
                anchor_windows[i,c],
                (out_size,out_size),
                interpolation=cv2.INTER_AREA,
            )
    return result

MULTIVIEW_SERIES_RESCUE_COUNT=0

def build_selected_s11_views(uid,selected,anchor_windows):
    global MULTIVIEW_SERIES_RESCUE_COUNT

    assert len(selected)==18
    assert anchor_windows.shape==(18,3,224,224)

    detail=np.empty((18,3,280,280),dtype=np.uint8)
    context=np.empty((18,3,224,224),dtype=np.uint8)
    audits=[]

    # DataFrame row order is the S9 nested selection order.
    selected=selected.reset_index(drop=True).copy()
    selected['_out_pos']=np.arange(18,dtype=np.int32)

    for sid,g in selected.groupby(SID,sort=False):
        sid=str(sid)
        positions=g['_out_pos'].astype(int).tolist()
        centers=g['center'].astype(int).tolist()
        unique_needed=sorted(set(
            j
            for c in centers
            for j in (c-1,c,c+1)
        ))

        try:
            raw,shape,spacing,diag=decode_series_raw_robust(str(uid),sid)

            # Normal hidden path: exact S11 training-cache view transforms.
            gd=no_pad_geometry_robust(
                shape,spacing,DETAIL_FOV_MM,DETAIL_SIZE
            )
            gcx=no_pad_geometry_robust(
                shape,spacing,CONTEXT_FOV_MM,CONTEXT_SIZE
            )

            ds,da=normalized_view_selected(raw,gd,unique_needed)
            cs,ca=normalized_view_selected(raw,gcx,unique_needed)

            dw=make_selected_triplets(ds,centers)
            cw=make_selected_triplets(cs,centers)

            for j,pos in enumerate(positions):
                detail[pos]=dw[j]
                context[pos]=cw[j]

            audits.append({
                UID:str(uid),SID:sid,
                'selected_windows':len(centers),
                'view_mode':'EXACT_S11_RAW_DICOM',
                'detail_coverage_min':gd['coverage_min'],
                'context_coverage_min':gcx['coverage_min'],
                'detail_geometry_fallback':gd['geometry_fallback'],
                'context_geometry_fallback':gcx['geometry_fallback'],
                'decode_errors':diag.get('multiview_decode_errors',0),
                'detail_degenerate':da['degenerate_intensity_fallback'],
                'context_degenerate':ca['degenerate_intensity_fallback'],
            })

        except Exception as e:
            # Real-image rescue only. No neutral prediction is introduced.
            MULTIVIEW_SERIES_RESCUE_COUNT+=1
            local=anchor_windows[np.asarray(positions,dtype=np.int64)]
            detail[np.asarray(positions)]=anchor_resize_fallback(local,280)
            context[np.asarray(positions)]=local
            audits.append({
                UID:str(uid),SID:sid,
                'selected_windows':len(centers),
                'view_mode':'RESCUE_FROM_REAL_ANCHOR',
                'error':repr(e)[:500],
            })

    assert detail.shape==(18,3,280,280)
    assert context.shape==(18,3,224,224)
    return detail,anchor_windows,context,audits

In [ ]:
# Dual-fold feature extraction, S8C direct/selector, and branch-level rescue.
# IMPORTANT: when both folds succeed, this reproduces V1 arithmetic exactly.

FEATURE_CHUNK=18
MEAN=[
    torch.tensor([.485,.456,.406],device=f'cuda:{f}').view(1,3,1,1)
    for f in (0,1)
]
STD=[
    torch.tensor([.229,.224,.225],device=f'cuda:{f}').view(1,3,1,1)
    for f in (0,1)
]

EXECUTOR=ThreadPoolExecutor(max_workers=2)
PARALLEL_FALLBACK_COUNT=0
FEATURE_FOLD_RETRY_COUNT=0
S8C_FOLD_RESCUE_COUNT=0
S8C_ALL_FOLD_FAIL_COUNT=0
S10_FOLD_RESCUE_COUNT=0
S10_ALL_FOLD_FAIL_COUNT=0


@torch.inference_mode()
def extract_features_fold(windows,fold):
    dev=torch.device(f'cuda:{fold}')
    enc=SELECTOR_ENCODERS[fold]
    parts=[]

    for b0 in range(0,len(windows),FEATURE_CHUNK):
        x=torch.from_numpy(
            windows[b0:b0+FEATURE_CHUNK]
        ).to(
            dev,
            dtype=torch.float32,
        ).div_(255.)

        with torch.autocast(
            'cuda',
            dtype=torch.float16,
        ):
            tok=enc(
                pixel_values=(x-MEAN[fold])/STD[fold],
                interpolate_pos_encoding=True,
                return_dict=True,
            ).last_hidden_state

            feat=torch.cat(
                [
                    tok[:,0],
                    tok[:,1:].mean(1),
                ],
                dim=-1,
            ).float()

        # Reproduce S7/S8 cache fp16 round-trip.
        feat=feat.half().float().cpu()
        parts.append(feat)

        del x,tok

    return torch.cat(parts,0)


def _retry_feature_fold(windows,fold,why):
    global FEATURE_FOLD_RETRY_COUNT
    FEATURE_FOLD_RETRY_COUNT+=1

    print(
        'FEATURE_FOLD_RETRY',
        'fold',fold,
        'reason',repr(why)[:300],
        flush=True,
    )

    gc.collect()
    torch.cuda.empty_cache()

    return extract_features_fold(
        windows,
        fold,
    )


def extract_features_two_folds(windows):
    """
    Normal path:
        same parallel Fold0/Fold1 extraction as V1.

    If the parallel executor itself fails, retry sequentially.
    A persistent fold failure is still raised here because S8C candidate rows
    must remain aligned across both folds during the primary path.
    """
    global PARALLEL_FALLBACK_COUNT

    try:
        fut=[
            EXECUTOR.submit(
                extract_features_fold,
                windows,
                f,
            )
            for f in (0,1)
        ]

        return [
            fut[0].result(),
            fut[1].result(),
        ]

    except Exception as e:
        PARALLEL_FALLBACK_COUNT+=1

        print(
            'DUAL_GPU_PARALLEL_FALLBACK',
            repr(e)[:500],
            flush=True,
        )

        gc.collect()
        torch.cuda.empty_cache()

        out=[]
        for f in (0,1):
            out.append(
                _retry_feature_fold(
                    windows,
                    f,
                    e,
                )
            )
        return out


@torch.inference_mode()
def score_s8c_fold(feature_series,fold):
    # Same S8C computation as V1.
    # One forward produces both:
    #   (1) full-MRI direct probability
    #   (2) exact signed per-window classifier contribution.
    model=S8C_MODELS[fold]
    dev=torch.device(f'cuda:{fold}')

    zs=[]
    was=[]
    pooled=[]

    for cpu in feature_series:
        x=cpu.to(
            dev,
            dtype=torch.float32,
            non_blocking=False,
        )

        z=model.proj(x)
        wa=torch.softmax(
            model.window_attn(z).float(),
            dim=0,
        ).to(z.dtype)

        zs.append(z)
        was.append(wa)

        pooled.append(
            torch.einsum(
                'wd,wt->td',
                z,
                wa,
            )
        )

        del x

    st=torch.stack(pooled)

    sl=torch.einsum(
        'std,td->st',
        st,
        model.series_query,
    )*model.series_scale

    sa=torch.softmax(
        sl.float(),
        dim=0,
    ).to(st.dtype)

    pool=torch.einsum(
        'std,st->td',
        st,
        sa,
    )

    direct_logits=(
        pool*model.head.weight
    ).sum(-1)+model.head.bias

    direct_prob=torch.sigmoid(
        direct_logits.float()
    ).cpu().numpy().astype(np.float64)

    contrib=[]

    for s,(z,wa) in enumerate(
        zip(zs,was)
    ):
        mass=wa*sa[s].view(1,-1)
        contrib.append(
            mass*(z@model.head.weight.T)
        )

    c=torch.cat(contrib)

    recon=float(
        (
            (c.sum(0)+model.head.bias)
            -direct_logits
        ).abs().max().cpu()
    )

    assert recon<2e-5,(fold,recon)
    assert direct_prob.shape==(12,)
    assert np.isfinite(direct_prob).all()
    assert (direct_prob>=0).all()
    assert (direct_prob<=1).all()

    return (
        c.float().cpu().numpy(),
        recon,
        direct_prob,
    )


def score_s8c_fold_safe(feature_series,fold):
    """
    Retry a single S8C fold once after CUDA cleanup.
    Return None instead of killing the Study if that fold remains unusable.
    """
    global S8C_FOLD_RESCUE_COUNT

    try:
        return score_s8c_fold(
            feature_series,
            fold,
        ),None

    except Exception as first:
        S8C_FOLD_RESCUE_COUNT+=1

        print(
            'S8C_FOLD_RETRY',
            'fold',fold,
            repr(first)[:350],
            flush=True,
        )

        gc.collect()
        torch.cuda.empty_cache()

        try:
            return score_s8c_fold(
                feature_series,
                fold,
            ),None

        except Exception as second:
            print(
                'S8C_FOLD_UNAVAILABLE',
                'fold',fold,
                repr(second)[:350],
                flush=True,
            )
            return None,{
                'fold':fold,
                'first':repr(first)[:500],
                'second':repr(second)[:500],
            }


def percentile_rank(v):
    return pd.Series(
        np.asarray(v,np.float64)
    ).rank(
        method='average',
        pct=True,
    ).to_numpy(np.float32)


def abs_ranks(c):
    a=np.abs(
        np.asarray(c,np.float32)
    )
    out=np.empty_like(a)

    for t in range(12):
        out[:,t]=percentile_rank(
            a[:,t]
        )

    return out


def sorted_df(df):
    return df.sort_values(
        [
            'selector_score',
            'selector_target_mean',
            SID,
            'center',
        ],
        ascending=[
            False,
            False,
            True,
            True,
        ],
        kind='mergesort',
    )


def nested_order(df,max_k=18,gap=3):
    ordered=sorted_df(df)
    chosen=set()
    sel=[]
    by={}

    def ok(r,g):
        return (
            g<=0
            or all(
                abs(int(r.center)-c)>=g
                for c in by.get(
                    str(r[SID]),
                    [],
                )
            )
        )

    def add(i,r):
        chosen.add(i)
        sel.append(i)
        by.setdefault(
            str(r[SID]),
            [],
        ).append(
            int(r.center)
        )

    seeds=sorted_df(
        ordered.groupby(
            SID,
            sort=False,
            as_index=False,
        ).head(1)
    )

    for i,r in seeds.iterrows():
        if len(sel)>=max_k:
            break
        add(i,r)

    for g in [gap,2,1,0]:
        if len(sel)>=max_k:
            break

        for i,r in ordered.iterrows():
            if i in chosen:
                continue

            if ok(r,g):
                add(i,r)

                if len(sel)>=max_k:
                    break

    return sel


def _pad_selected(frame,selected):
    padded=0

    if len(selected)<TOPK:
        ordered=sorted_df(frame)

        if len(ordered)==0:
            raise RuntimeError(
                'No candidate windows for Top18'
            )

        rows=[
            r
            for _,r in selected.iterrows()
        ] if len(selected) else []

        j=0

        while len(rows)<TOPK:
            rows.append(
                ordered.iloc[
                    j%len(ordered)
                ]
            )
            j+=1
            padded+=1

        selected=pd.DataFrame(
            rows
        ).reset_index(
            drop=True
        )

    assert len(selected)==18

    return selected,padded


def deterministic_coverage_top18(meta):
    """
    Emergency selector used ONLY if both S8C folds fail after retry.

    It is target/label-free:
    - prioritizes the center of each surviving Series,
    - preserves Series coverage first,
    - then applies the same center-gap relaxation.

    This path never runs on a normal Study.
    """
    frame=pd.DataFrame({
        SID:[m['sid'] for m in meta],
        'center':[int(m['center']) for m in meta],
        'row_idx':np.arange(
            len(meta),
            dtype=np.int32,
        ),
    })

    if len(frame)==0:
        raise RuntimeError(
            'No windows for emergency Top18'
        )

    score=np.zeros(
        len(frame),
        dtype=np.float32,
    )

    for _,idx in frame.groupby(
        SID,
        sort=False,
    ).groups.items():
        idx=np.asarray(
            list(idx),
            dtype=np.int64,
        )

        centers=frame.loc[
            idx,
            'center',
        ].to_numpy(np.float32)

        med=float(
            np.median(centers)
        )
        span=max(
            1.0,
            float(
                centers.max()
                -centers.min()
            ),
        )

        score[idx]=(
            1.0
            -np.minimum(
                1.0,
                np.abs(
                    centers-med
                )/(span/2.0+1e-6),
            )
        )

    frame['selector_score']=score
    frame['selector_target_mean']=score
    frame['best_target']='EMERGENCY_COVERAGE'

    sel=nested_order(
        frame,
        TOPK,
        MIN_GAP,
    )

    selected=frame.loc[
        sel
    ].copy().reset_index(
        drop=True
    )

    selected,padded=_pad_selected(
        frame,
        selected,
    )

    return selected,padded


def select_top18_and_s8c_direct(meta,features_by_fold):
    """
    Normal case (2/2 S8C folds alive):
      EXACT V1 math.

    Rescue case (1/2 folds alive):
      use that surviving fold's contribution rank and direct probability.

    Emergency case (0/2 folds alive):
      target-free deterministic coverage Top18,
      no S8C direct branch.
    """
    global S8C_ALL_FOLD_FAIL_COUNT

    ranks=[]
    recons=[]
    direct_fold_probs=[]
    alive=[]
    errors=[]

    for f in (0,1):
        scored,err=score_s8c_fold_safe(
            features_by_fold[f],
            f,
        )

        if scored is None:
            errors.append(err)
            continue

        c,recon,direct_prob=scored

        ranks.append(
            abs_ranks(c)
        )
        recons.append(
            (f,recon)
        )
        direct_fold_probs.append(
            direct_prob
        )
        alive.append(f)

    if alive:
        R=np.mean(
            ranks,
            axis=0,
        ).astype(np.float32)

        score=R.max(1)
        tm=R.mean(1)
        best=np.argmax(
            R,
            axis=1,
        )

        frame=pd.DataFrame({
            SID:[m['sid'] for m in meta],
            'center':[m['center'] for m in meta],
            'row_idx':np.arange(
                len(meta),
                dtype=np.int32,
            ),
            'selector_score':score,
            'selector_target_mean':tm,
            'best_target':[
                TARGETS[j]
                for j in best
            ],
        })

        sel=nested_order(
            frame,
            TOPK,
            MIN_GAP,
        )

        selected=frame.loc[
            sel
        ].copy().reset_index(
            drop=True
        )

        selected,padded=_pad_selected(
            frame,
            selected,
        )

        direct_fold_probs=np.stack(
            direct_fold_probs,
            axis=0,
        )

        p_s8c_direct=direct_fold_probs.mean(
            axis=0
        ).astype(np.float64)

        assert p_s8c_direct.shape==(12,)
        assert np.isfinite(
            p_s8c_direct
        ).all()

        return {
            'selected':selected,
            'recons':recons,
            'padded':padded,
            'p_s8c_direct':p_s8c_direct,
            's8c_fold_probs':direct_fold_probs,
            's8c_alive_folds':alive,
            's8c_errors':errors,
            'selector_mode':(
                'S8C_FOLD01_CONSENSUS'
                if len(alive)==2
                else f'S8C_SINGLE_FOLD_{alive[0]}'
            ),
        }

    # Both S8C folds unavailable after retry.
    S8C_ALL_FOLD_FAIL_COUNT+=1

    selected,padded=deterministic_coverage_top18(
        meta
    )

    print(
        'S8C_ALL_FOLDS_FAILED_EMERGENCY_TOP18',
        flush=True,
    )

    return {
        'selected':selected,
        'recons':[],
        'padded':padded,
        'p_s8c_direct':None,
        's8c_fold_probs':None,
        's8c_alive_folds':[],
        's8c_errors':errors,
        'selector_mode':'EMERGENCY_COVERAGE_TOP18',
    }

# ------------------------------------------------------------------
# S11 LAST 2F inference.
# ------------------------------------------------------------------
MEAN_S11=[
    torch.tensor([.485,.456,.406],device=f'cuda:{f}').view(1,1,3,1,1)
    for f in (0,1)
]
STD_S11=[
    torch.tensor([.229,.224,.225],device=f'cuda:{f}').view(1,1,3,1,1)
    for f in (0,1)
]

S11_FOLD_RETRY_COUNT=0
S11_ALL_FOLD_FAIL_COUNT=0

def prep_s11(x,fold):
    return (
        torch.from_numpy(np.ascontiguousarray(x))
        .to(torch.device(f'cuda:{fold}'),dtype=torch.float32)
        .div_(255.)
        -MEAN_S11[fold]
    )/STD_S11[fold]

@torch.inference_mode()
def s11_predict_fold(detail,anchor,context,fold):
    d=prep_s11(detail[None],fold)
    a=prep_s11(anchor[None],fold)
    c=prep_s11(context[None],fold)

    with torch.autocast('cuda',dtype=torch.float16):
        logits=S11_MODELS[fold](
            d,a,c,chunk=18
        )

    p=torch.sigmoid(
        logits
    ).float().cpu().numpy()[0].astype(np.float64)

    assert p.shape==(12,)
    assert np.isfinite(p).all()
    assert (p>=0).all() and (p<=1).all()
    return p

def s11_predict_fold_safe(detail,anchor,context,fold):
    global S11_FOLD_RETRY_COUNT
    try:
        return s11_predict_fold(
            detail,anchor,context,fold
        ),None
    except Exception as first:
        S11_FOLD_RETRY_COUNT+=1
        print(
            'S11_FOLD_RETRY',
            'fold',fold,
            repr(first)[:350],
            flush=True,
        )
        gc.collect()
        torch.cuda.empty_cache()
        try:
            return s11_predict_fold(
                detail,anchor,context,fold
            ),None
        except Exception as second:
            print(
                'S11_FOLD_UNAVAILABLE',
                'fold',fold,
                repr(second)[:350],
                flush=True,
            )
            return None,{
                'fold':fold,
                'first':repr(first)[:500],
                'second':repr(second)[:500],
            }

In [ ]:
# Hidden-test execution: S11 LAST 70% + S8C Full-MRI Direct 30%.
# No Study-level 0.5 placeholder is ever written.

BRANCH_RESCUE_COUNT=0
STUDY_EXCEPTION_COUNT=0

def process_study(uid):
    global S11_ALL_FOLD_FAIL_COUNT,BRANCH_RESCUE_COUNT

    rows=TS.loc[
        TS[UID].astype(str)==str(uid)
    ].sort_values(
        SID,kind='stable'
    ).reset_index(drop=True)

    if rows.empty:
        raise RuntimeError(f'No test series found for {uid}')

    raw_series=[]
    feature_series={0:[],1:[]}
    meta=[]
    sdiags=[]
    failed=0

    for _,row in rows.iterrows():
        sid=str(row[SID])
        try:
            windows,centers,diag=build_series_windows(
                str(uid),sid
            )
            f0,f1=extract_features_two_folds(windows)
            assert (
                len(windows)==len(centers)
                ==len(f0)==len(f1)
            )

            raw_series.append(windows)
            feature_series[0].append(f0)
            feature_series[1].append(f1)
            meta.extend([
                {'sid':sid,'center':int(c)}
                for c in centers
            ])
            sdiags.append(diag)

        except Exception as e:
            failed+=1
            sdiags.append({
                UID:str(uid),SID:sid,
                'series_failed':1,
                'error':repr(e)[:500],
            })
            print(
                'SERIES_SKIPPED',
                uid,sid,repr(e)[:500],
                flush=True,
            )
            gc.collect()
            torch.cuda.empty_cache()

    if not raw_series:
        raise RuntimeError(f'All Series failed for Study {uid}')

    anchor_candidates=np.concatenate(raw_series,axis=0)
    assert len(anchor_candidates)==len(meta)
    assert len(anchor_candidates)==sum(
        len(x) for x in feature_series[0]
    )
    assert len(anchor_candidates)==sum(
        len(x) for x in feature_series[1]
    )

    # Same hidden-tested S8C forward produces BOTH:
    # 1) Full-MRI Direct probability
    # 2) label-free Top18 selector contribution.
    s8=select_top18_and_s8c_direct(
        meta,feature_series
    )
    selected=s8['selected'].copy().reset_index(drop=True)

    idx=selected.row_idx.astype(int).to_numpy()
    anchor=np.ascontiguousarray(
        anchor_candidates[idx],
        dtype=np.uint8,
    )
    assert anchor.shape==(18,3,224,224)

    # Same S11 standalone Multi-View reconstruction.
    detail,anchor,context,mvdiag=build_selected_s11_views(
        uid,selected,anchor
    )

    s11_probs=[]
    s11_alive=[]
    s11_errors=[]

    for f in (0,1):
        p,err=s11_predict_fold_safe(
            detail,anchor,context,f
        )
        if p is not None:
            s11_probs.append(p)
            s11_alive.append(f)
        else:
            s11_errors.append(err)

    p_s11=None
    s11_fold_probs=None
    if s11_probs:
        s11_fold_probs=np.stack(s11_probs,axis=0)
        p_s11=s11_fold_probs.mean(axis=0).astype(np.float64)
        assert p_s11.shape==(12,)
        assert np.isfinite(p_s11).all()
        assert (p_s11>=0).all() and (p_s11<=1).all()
    else:
        S11_ALL_FOLD_FAIL_COUNT+=1

    p_s8c_direct=s8['p_s8c_direct']
    if p_s8c_direct is not None:
        p_s8c_direct=np.asarray(p_s8c_direct,np.float64)
        assert p_s8c_direct.shape==(12,)
        assert np.isfinite(p_s8c_direct).all()
        assert (p_s8c_direct>=0).all() and (p_s8c_direct<=1).all()

    # --------------------------------------------------------------
    # Normal path:
    #   S11 2F alive + S8C 2F alive
    #   -> exact 70:30 probability blend.
    #
    # Rescue:
    #   both branches have real predictions but one branch is 1F
    #   -> same 70:30 weights with each surviving branch estimate.
    #   only one branch remains -> use that real branch 100%.
    #   no real branch -> fatal.
    # --------------------------------------------------------------
    if p_s11 is not None and p_s8c_direct is not None:
        p_final=(
            BLEND_S11*p_s11
            +BLEND_S8C*p_s8c_direct
        ).astype(np.float64)

        normal=(
            len(s11_alive)==2
            and len(s8['s8c_alive_folds'])==2
        )
        blend_mode=(
            'NORMAL_S11_LAST_70_S8C_DIRECT_30'
            if normal
            else 'RESCUE_70_30_WITH_SURVIVING_FOLDS'
        )
        if not normal:
            BRANCH_RESCUE_COUNT+=1

    elif p_s11 is not None:
        p_final=p_s11.copy()
        blend_mode='RESCUE_S11_ONLY_100'
        BRANCH_RESCUE_COUNT+=1

    elif p_s8c_direct is not None:
        p_final=p_s8c_direct.copy()
        blend_mode='RESCUE_S8C_DIRECT_ONLY_100'
        BRANCH_RESCUE_COUNT+=1

    else:
        raise RuntimeError(
            f'No valid real-image prediction branch for Study {uid}'
        )

    assert p_final.shape==(12,)
    assert np.isfinite(p_final).all()
    assert (p_final>=0).all() and (p_final<=1).all()

    s11_fold_mad=(
        float(np.abs(
            s11_fold_probs[0]-s11_fold_probs[1]
        ).mean())
        if s11_fold_probs is not None and len(s11_fold_probs)>=2
        else None
    )

    s8c_fold_probs=s8['s8c_fold_probs']
    s8c_fold_mad=(
        float(np.abs(
            s8c_fold_probs[0]-s8c_fold_probs[1]
        ).mean())
        if s8c_fold_probs is not None and len(s8c_fold_probs)>=2
        else None
    )

    cross_mad=(
        float(np.abs(p_s11-p_s8c_direct).mean())
        if p_s11 is not None and p_s8c_direct is not None
        else None
    )

    diag={
        UID:str(uid),
        'listed_series':len(rows),
        'used_series':len(raw_series),
        'series_failed':failed,
        'candidate_windows':len(anchor_candidates),
        'top18_padding':int(s8['padded']),
        'selector_mode':s8['selector_mode'],
        's8c_alive_folds':','.join(map(str,s8['s8c_alive_folds'])),
        's11_alive_folds':','.join(map(str,s11_alive)),
        'blend_mode':blend_mode,
        'selector_recon_max':(
            max(x[1] for x in s8['recons'])
            if s8['recons'] else None
        ),
        's11_fold_mad':s11_fold_mad,
        's8c_direct_fold_mad':s8c_fold_mad,
        's11_vs_s8c_direct_mad':cross_mad,
        's11_prob_min':(
            float(p_s11.min()) if p_s11 is not None else None
        ),
        's11_prob_max':(
            float(p_s11.max()) if p_s11 is not None else None
        ),
        's8c_direct_prob_min':(
            float(p_s8c_direct.min()) if p_s8c_direct is not None else None
        ),
        's8c_direct_prob_max':(
            float(p_s8c_direct.max()) if p_s8c_direct is not None else None
        ),
        'blend_prob_min':float(p_final.min()),
        'blend_prob_max':float(p_final.max()),
        'multiview_rescued_series':int(
            sum(x.get('view_mode')=='RESCUE_FROM_REAL_ANCHOR' for x in mvdiag)
        ),
    }

    return (
        p_final,
        p_s11,
        p_s8c_direct,
        diag,
        sdiags,
        mvdiag,
        selected,
        s11_fold_probs,
        s8c_fold_probs,
        s11_alive,
        s8['s8c_alive_folds'],
    )


submission_path=OUT/'submission.csv'
if submission_path.exists():
    submission_path.unlink()

uids=SAMPLE[UID].astype(str).tolist()
final_rows=[]
s11_rows=[]
s8c_rows=[]
s11_fold0_rows=[]
s11_fold1_rows=[]
s8c_fold0_rows=[]
s8c_fold1_rows=[]
study_diag=[]
series_diag=[]
multiview_diag=[]
selection_rows=[]

start=time.monotonic()

for i,uid in enumerate(uids,1):
    try:
        (
            p,
            p_s11,
            p_s8c,
            diag,
            sdiags,
            mvdiag,
            sel,
            s11_fold_probs,
            s8c_fold_probs,
            s11_alive,
            s8c_alive,
        )=process_study(uid)

        final_rows.append({
            UID:uid,
            **{t:float(p[j]) for j,t in enumerate(TARGETS)}
        })

        s11_rows.append({
            UID:uid,
            **{
                t:(float(p_s11[j]) if p_s11 is not None else np.nan)
                for j,t in enumerate(TARGETS)
            }
        })
        s8c_rows.append({
            UID:uid,
            **{
                t:(float(p_s8c[j]) if p_s8c is not None else np.nan)
                for j,t in enumerate(TARGETS)
            }
        })

        s11_by_fold=(
            {s11_alive[j]:s11_fold_probs[j] for j in range(len(s11_alive))}
            if s11_fold_probs is not None else {}
        )
        s8c_by_fold=(
            {s8c_alive[j]:s8c_fold_probs[j] for j in range(len(s8c_alive))}
            if s8c_fold_probs is not None else {}
        )

        for fold,store,by in [
            (0,s11_fold0_rows,s11_by_fold),
            (1,s11_fold1_rows,s11_by_fold),
        ]:
            store.append({
                UID:uid,
                **{
                    t:(float(by[fold][j]) if fold in by else np.nan)
                    for j,t in enumerate(TARGETS)
                }
            })

        for fold,store,by in [
            (0,s8c_fold0_rows,s8c_by_fold),
            (1,s8c_fold1_rows,s8c_by_fold),
        ]:
            store.append({
                UID:uid,
                **{
                    t:(float(by[fold][j]) if fold in by else np.nan)
                    for j,t in enumerate(TARGETS)
                }
            })

        study_diag.append(diag)
        series_diag.extend(sdiags)
        multiview_diag.extend(mvdiag)

        q=sel.copy()
        q.insert(0,UID,uid)
        q.insert(1,'selection_order',np.arange(1,19))
        selection_rows.append(q)

    except Exception as e:
        STUDY_EXCEPTION_COUNT+=1
        fatal={
            'index':i,
            UID:uid,
            'error':repr(e),
            'trace':__import__('traceback').format_exc()[-8000:],
            'action':'FATAL_NO_REAL_BLEND_PREDICTION',
        }
        print(
            'FATAL_NO_REAL_BLEND_PREDICTION',
            json.dumps(fatal,ensure_ascii=False),
            flush=True,
        )

        if study_diag:
            pd.DataFrame(study_diag).to_csv(
                AUDIT/'study_diagnostics_partial.csv',
                index=False,
            )
        if final_rows:
            pd.DataFrame(final_rows).to_csv(
                AUDIT/'partial_real_predictions.csv',
                index=False,
            )

        gc.collect()
        torch.cuda.empty_cache()
        raise RuntimeError(
            'FATAL_NO_REAL_BLEND_PREDICTION: '
            f'Study {uid} has no valid real-image prediction'
        ) from e

    if i%10==0 or i==len(uids):
        pd.DataFrame(final_rows).to_csv(
            AUDIT/'partial_real_predictions.csv',
            index=False,
        )
        print(
            'INFERENCE_PROGRESS',
            i,'/',len(uids),
            'mode',study_diag[-1]['blend_mode'],
            'branch_rescues',BRANCH_RESCUE_COUNT,
            's11_fold_rescues',S11_FOLD_RETRY_COUNT,
            's8c_fold_rescues',S8C_FOLD_RESCUE_COUNT,
            'mv_series_rescues',MULTIVIEW_SERIES_RESCUE_COUNT,
            'elapsed_min',round((time.monotonic()-start)/60,2),
            flush=True,
        )

final_df=pd.DataFrame(final_rows)
s11_df=pd.DataFrame(s11_rows)
s8c_df=pd.DataFrame(s8c_rows)

assert len(final_df)==len(SAMPLE)
assert final_df[UID].is_unique
assert final_df[UID].tolist()==uids

submission=SAMPLE[[UID]].merge(
    final_df,on=UID,how='left',validate='one_to_one'
)
assert submission[UID].tolist()==uids
assert list(submission.columns)==[UID]+TARGETS

values=submission[TARGETS].to_numpy(np.float64)
assert np.isfinite(values).all()
assert (values>=0).all() and (values<=1).all()

submission.to_csv(submission_path,index=False)

# Branch/fold predictions for exact arithmetic QA.
s11_df.to_csv(AUDIT/'s11_last_2f_branch_predictions.csv',index=False)
s8c_df.to_csv(AUDIT/'s8c_direct_2f_branch_predictions.csv',index=False)
pd.DataFrame(s11_fold0_rows).to_csv(AUDIT/'s11_fold0_predictions.csv',index=False)
pd.DataFrame(s11_fold1_rows).to_csv(AUDIT/'s11_fold1_predictions.csv',index=False)
pd.DataFrame(s8c_fold0_rows).to_csv(AUDIT/'s8c_fold0_direct_predictions.csv',index=False)
pd.DataFrame(s8c_fold1_rows).to_csv(AUDIT/'s8c_fold1_direct_predictions.csv',index=False)

pd.DataFrame(study_diag).to_csv(
    AUDIT/'study_diagnostics.csv',index=False
)
pd.DataFrame(series_diag).to_csv(
    AUDIT/'series_diagnostics.csv',index=False
)
pd.DataFrame(multiview_diag).to_csv(
    AUDIT/'multiview_diagnostics.csv',index=False
)
if selection_rows:
    pd.concat(
        selection_rows,ignore_index=True
    ).to_csv(
        AUDIT/'hidden_test_top18_manifest.csv',
        index=False,
    )

# --------------------------------------------------------------
# Exact normal-path 70:30 arithmetic QA.
# --------------------------------------------------------------
diag_df=pd.DataFrame(study_diag)
normal_mode='NORMAL_S11_LAST_70_S8C_DIRECT_30'
normal_uids=set(
    diag_df.loc[
        diag_df['blend_mode']==normal_mode,
        UID,
    ].astype(str)
)

max_normal_blend_error=0.0

if normal_uids:
    fidx=final_df.set_index(UID)
    sidx=s11_df.set_index(UID)
    didx=s8c_df.set_index(UID)
    order=[u for u in uids if u in normal_uids]

    expected=(
        BLEND_S11*sidx.loc[order,TARGETS].to_numpy(np.float64)
        +BLEND_S8C*didx.loc[order,TARGETS].to_numpy(np.float64)
    )
    observed=fidx.loc[order,TARGETS].to_numpy(np.float64)

    max_normal_blend_error=float(
        np.max(np.abs(expected-observed))
    )
    assert max_normal_blend_error<1e-12

report={
    'status':'SUBMISSION_SUCCESS',
    'experiment':EXPERIMENT,
    'public_lb_reference':{
        'S10_standalone':0.923,
        'S10_70_S8C_30':0.926,
        'S11_standalone_user_reported':0.926,
    },
    'normal_formula':'0.70*S11_LAST_2F + 0.30*S8C_FULL_MRI_DIRECT_2F',
    'blend_weights':{
        'S11_LAST':BLEND_S11,
        'S8C_DIRECT':BLEND_S8C,
    },
    'studies':len(submission),
    'test_series_rows':len(TS),
    'models':MODEL_AUDIT,
    'runtime_minutes':(time.monotonic()-start)/60,
    'submission_sha256':sha256(submission_path),
    'normal_70_30_studies':int(
        (diag_df['blend_mode']==normal_mode).sum()
    ),
    'branch_rescue_studies':int(BRANCH_RESCUE_COUNT),
    'study_level_0p5_fallbacks':0,
    'study_exception_count':int(STUDY_EXCEPTION_COUNT),
    'parallel_feature_fallbacks':int(PARALLEL_FALLBACK_COUNT),
    'feature_fold_retries':int(FEATURE_FOLD_RETRY_COUNT),
    's8c_fold_retries':int(S8C_FOLD_RESCUE_COUNT),
    's8c_all_fold_failures':int(S8C_ALL_FOLD_FAIL_COUNT),
    's11_fold_retries':int(S11_FOLD_RETRY_COUNT),
    's11_all_fold_failures':int(S11_ALL_FOLD_FAIL_COUNT),
    'multiview_series_rescues':int(MULTIVIEW_SERIES_RESCUE_COUNT),
    'max_normal_blend_arithmetic_error':float(max_normal_blend_error),
    'preprocess':{
        'selector_anchor':'RMIL04B_NATIVE224_V3_ROBUST',
        'detail':'110MM_NO_PAD_SERIES_P005_995_ZSCORE_CLAMP5_AREA280_UINT8',
        'anchor':'130MM_NATIVE224_V3',
        'context':'160MM_NO_PAD_SERIES_P005_995_ZSCORE_CLAMP5_AREA224_UINT8',
        'routing':'FIXED_TARGET_WISE_LOGIT_ROUTING',
    },
    'rescue_policy':[
        'DICOM IOP/IPP -> InstanceNumber -> filename',
        'nearest valid decoded slice',
        '1-2 slice edge replication',
        'failed candidate Series skipped',
        'dual-GPU selector feature sequential retry',
        'S8C 2F -> 1F -> deterministic label-free emergency Top18',
        'selected-view failure -> real ANCHOR-derived view rescue',
        'S11 2F -> 1F retry/rescue',
        'if one whole prediction branch unavailable -> surviving real branch 100%',
        'NO Study-level 0.5 fallback',
    ],
}

(AUDIT/'submission_report.json').write_text(
    json.dumps(report,indent=2,ensure_ascii=False),
    encoding='utf-8',
)

review=OUT/'S11_LAST_70_S8C_DIRECT_30_LB_V1_REVIEW.zip'
with zipfile.ZipFile(
    review,'w',zipfile.ZIP_DEFLATED
) as z:
    for p in AUDIT.rglob('*'):
        if p.is_file():
            z.write(p,p.relative_to(AUDIT))

EXECUTOR.shutdown(wait=True)

print('SUBMISSION_SUCCESS',json.dumps(report,indent=2,ensure_ascii=False))
print('SUBMISSION_PATH',submission_path)
print('NORMAL_FORMULA','0.70*S11_LAST_2F + 0.30*S8C_FULL_MRI_DIRECT_2F')
print('MAX_NORMAL_BLEND_ARITHMETIC_ERROR',max_normal_blend_error)
print('STUDY_LEVEL_0P5_FALLBACKS',0)
print('REVIEW_ZIP',review)